In [31]:
import pandas as pd

In [32]:
df = pd.read_parquet("../datasets/coco-train-00000-of-00001.parquet")

In [33]:
df.head()

,caption1,caption2
0,A clock that blends in with the wall hangs in ...,A very clean and well decorated empty bathroom
1,A very clean and well decorated empty bathroom,A bathroom with a border of butterflies and bl...
2,A bathroom with a border of butterflies and bl...,An angled view of a beautifully decorated bath...
3,An angled view of a beautifully decorated bath...,A blue and white bathroom with butterfly theme...
4,A blue and white bathroom with butterfly theme...,A clock that blends in with the wall hangs in ...


In [34]:
df['caption1'].isna().any() or df['caption2'].isna().any()

np.False_

#### What we want to do
1. we want to store these captions somewhere (like a vector DB), that supports metadata (because we'll enrich these later like add the image link etc)
2. after storage, we want to try to query it

#### llamaindex
1. llamaindex uses documents->nodes[]

In [35]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter


In [36]:
documents : list[Document] = []
#for i, row in df.head(10).iterrows():
for i, row in df.iterrows():
  doc = Document(
    text="", # we'll not use Document.text for now
    metadata={
      "row_id": int(i),
      "caption1": row["caption1"],
      "caption2": row["caption2"], 
    }
  )
  documents.append(doc)

print(f"Loaded {len(documents)} Documents.")
documents[0]

Loaded 414010 Documents.


Document(id_='3e2aa19f-1fd7-4b0f-978d-08741cac85ae', embedding=None, metadata={'row_id': 0, 'caption1': 'A clock that blends in with the wall hangs in a bathroom. ', 'caption2': 'A very clean and well decorated empty bathroom'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=None, image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}')

#### Lets create our nodes
solution below temporarily uses `Document` because `node_parser` uses `Document.text`

In [37]:
# convert each caption into a node (temporarily use a Document for node_parser)
caption_docs: list[Document] = []
for d in documents:
  for slot in ('caption1', 'caption2'):
    cap = d.metadata.get(slot)
    caption_docs.append(
      Document(
        text=str(cap),
        metadata={
          "row_id": d.metadata['row_id'],
          "caption_slot": slot
        }
      )
    )

# SentenceSplitter is overkill for captions, but it gives us Nodes cleanly.
# chunk_size is large so each caption stays as one node.
node_parser = SentenceSplitter(chunk_size=10_000, chunk_overlap=0)
nodes = node_parser.get_nodes_from_documents(caption_docs)
print(f"Created {len(nodes)} Nodes.")


Created 828020 Nodes.


#### Lets try to create embeddings
(but for llamaindex, we don't need to do this verbose way, we just need to plug the embedding model)

In [38]:
from llama_index.embeddings.ollama import OllamaEmbedding

OLLAMA_EMBED_MODEL = "embeddinggemma"
embed_model = OllamaEmbedding(
  model_name=OLLAMA_EMBED_MODEL, 
  embed_batch_size=1024
)



In [39]:
texts = [n.get_content() for n in nodes]
texts[:5]


['A clock that blends in with the wall hangs in a bathroom.',
 'A very clean and well decorated empty bathroom',
 'A very clean and well decorated empty bathroom',
 'A bathroom with a border of butterflies and blue paint on the walls above it.',
 'A bathroom with a border of butterflies and blue paint on the walls above it.']

In [40]:
vectors = embed_model.get_text_embedding_batch(texts[:100])  # embed first 100 for testing
print(f"Embedded {len(vectors)} Nodes.")
print("Example:")
print(" - node text:", nodes[0].get_content())
print(" - node meta:", nodes[0].metadata)
print(" - vector dim:", len(vectors[0]))
vectors[0][:5]  # print first 5 dimensions of first vector

Embedded 100 Nodes.
Example:
 - node text: A clock that blends in with the wall hangs in a bathroom.
 - node meta: {'row_id': 0, 'caption_slot': 'caption1'}
 - vector dim: 768


[-0.15663518, -0.037190415, 0.04024828, 0.0014637744, -0.03341323]

In [51]:
# Lets build index
from llama_index.core import Settings, VectorStoreIndex, StorageContext, load_index_from_storage

Settings.embed_model = embed_model
nodes_slice=nodes[:100_000]

In [52]:
# build an index
index = VectorStoreIndex(nodes_slice)

In [53]:
nodes_slice_len=len(nodes_slice)
index_dir_name=f"vector_index_coco_{nodes_slice_len}_nodes_ollama_{OLLAMA_EMBED_MODEL}"

In [54]:
# persist
index.storage_context.persist(f"../storage/{index_dir_name}")

In [55]:
# load storage context from disk
storage_context_full_dir = f"../storage/{index_dir_name}"
print(storage_context_full_dir)
storage_context = StorageContext.from_defaults(persist_dir=storage_context_full_dir)

../storage/vector_index_coco_100000_nodes_ollama_embeddinggemma


In [56]:
# build index from storage context
# these steps are not needed if you have the index object already
index = load_index_from_storage(storage_context)

In [57]:
# retriever returns nodes ranked by similarity
retriever = index.as_retriever(similarity_top_k=5)
query = "cat superman"
hits = retriever.retrieve(query)

for i,h in enumerate(hits,1):
  n = h.node
  print(f"{i:02d}. score={h.score:.4f} row_id={n.metadata.get('row_id')} slot={n.metadata.get('caption_slot')}")
  print(f"    {n.get_content().strip()}")


01. score=0.5360 row_id=29855 slot=caption1
    a couple of cats are around a person
02. score=0.5319 row_id=22583 slot=caption1
    a cat is kneeling on top of a car
03. score=0.5309 row_id=15138 slot=caption1
    black and white cat with a plastic tunnel watching a man
04. score=0.5285 row_id=15137 slot=caption2
    black and white cat with a plastic tunnel watching a man
05. score=0.5284 row_id=27287 slot=caption2
    A cat is sleeping across her owners chest


In [58]:
query = "dog riding a bike"
hits = retriever.retrieve(query)

for i,h in enumerate(hits,1):
  n = h.node
  print(f"{i:02d}. score={h.score:.4f} row_id={n.metadata.get('row_id')} slot={n.metadata.get('caption_slot')}")
  print(f"    {n.get_content().strip()}")

01. score=0.7327 row_id=6516 slot=caption1
    A dog rides on the back of a bicycle.
02. score=0.7244 row_id=6515 slot=caption2
    A dog rides on the back of a bicycle.
03. score=0.7218 row_id=21514 slot=caption1
    A small dog that is riding a bicycle.
04. score=0.7163 row_id=21513 slot=caption2
    A small dog that is riding a bicycle.
05. score=0.7122 row_id=21512 slot=caption1
    A dog riding on a tiny littel bike with training wheels.


#### lets try query engine with ollama 

In [59]:
from llama_index.llms.ollama import Ollama

OLLAMA_LLM_MODEL = "gemma3:1b"
Settings.llm = Ollama(
    model=OLLAMA_LLM_MODEL,
    request_timeout=120.0,  # if your model is slow
)

In [60]:
query_engine = index.as_query_engine(
    similarity_top_k=8,
    response_mode="compact",
)

query = "A dog playing frisbee in a park. Summarize the most relevant captions."
response = query_engine.query(query)

print("Answer:\n", str(response))

print("\nSources:")
for i, sn in enumerate(response.source_nodes, 1):
    meta = sn.node.metadata or {}
    print(f"{i}. score={sn.score:.4f} row_id={meta.get('row_id')} slot={meta.get('caption_slot')}")
    print(f"   {sn.node.get_content().strip()}")

Answer:
 A dog is in the grass trying to catch a frisbee.

Sources:
1. score=0.6959 row_id=28930 slot=caption1
   A small dog is playing outside trying to get the frisbee
2. score=0.6930 row_id=28929 slot=caption2
   A small dog is playing outside trying to get the frisbee
3. score=0.6844 row_id=8183 slot=caption1
   A dog is in the grass trying to catch a frisbee.
4. score=0.6792 row_id=8182 slot=caption2
   A dog is in the grass trying to catch a frisbee.
5. score=0.6756 row_id=8185 slot=caption1
   A dog is jumping in the air to catch a frisbee.
6. score=0.6723 row_id=8184 slot=caption2
   A dog is jumping in the air to catch a frisbee.
7. score=0.6671 row_id=28928 slot=caption2
   a small dog is cathing a frisbee in a field
8. score=0.6668 row_id=28929 slot=caption1
   a small dog is cathing a frisbee in a field
